# VNAI — ASR + MT (dịch) trên GPU (Colab) qua Cloudflare Tunnel

Notebook host trên GPU Colab:
- **ASR** (whisper) qua WebSocket `/asr`.
- **MT/dịch** (Ollama) qua proxy `/api/*` (cùng server, cùng tunnel).

Expose ra internet bằng **cloudflared** (Cloudflare quick tunnel) — free, **không cần token**, hỗ trợ WebSocket + streaming chunked ổn định.

## Cần chuẩn bị
- Runtime > Change runtime type > **T4 GPU**.
- Không cần tài khoản.

## Chọn model dịch (sửa biến `MT_MODEL_NAME` ở cell 6)
| Model | Size (q4) | Tốc độ T4 | Chất lượng vi-en | Ghi chú |
|---|---|---|---|---|
| `qwen2.5:7b` (mặc định) | ~4.5GB | khá | tốt | cân bằng tốt nhất |
| `qwen2.5:3b` | ~2GB | rất nhanh | khá | nhẹ, nhanh |
| `gemma3:4b` | ~2.6GB | nhanh | tốt (đa ngôn ngữ) | Google Gemma 3 |
| `qwen2.5:14b` | ~9GB | chậm hơn | rất tốt | nếu ưu tiên chất lượng |

Chạy tuần tự cell 1→7. Cell 5 chạy lại đến khi thấy `ASR ready`. Cell 6 (Ollama) pull model lần đầu ~2-3 phút.

In [ ]:
#@title 1. Kiểm tra GPU
import torch
if not torch.cuda.is_available():
    raise SystemExit('Chưa bật GPU. Runtime > Change runtime type > T4 GPU rồi chạy lại.')
print('GPU:', torch.cuda.get_device_name(0))
print('VRAM (GB):', round(torch.cuda.get_device_properties(0).total_memory/1e9, 1))

In [ ]:
#@title 2. Cài deps
import sys, subprocess
def pip(*a): subprocess.check_call([sys.executable,'-m','pip','install','-q',*a])
pip('fastapi[standard]','uvicorn[standard]','numpy','faster-whisper','httpx')
print('Installed.')

In [ ]:
#@title 3. Viết server ASR + proxy Ollama (asr_server.py)
server_code = r'''
import asyncio, json, logging
import numpy as np
import httpx
from contextlib import asynccontextmanager
from fastapi import FastAPI, WebSocket, WebSocketDisconnect, Request
from fastapi.responses import StreamingResponse

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger("asr_server")

model = None
OLLAMA_URL = "http://127.0.0.1:11434"

@asynccontextmanager
async def lifespan(app):
    global model
    from faster_whisper import WhisperModel
    logger.info("Loading whisper-medium on GPU (float16)...")
    model = WhisperModel("medium", device="cuda", compute_type="float16")
    logger.info("ASR ready.")
    app.state.ready = True
    yield
    logger.info("Shutting down ASR server...")

app = FastAPI(lifespan=lifespan)
app.state.ready = False

INITIAL_TRANSCRIBE_BYTES = 4800

async def _transcribe(audio_bytes, lang, beam_size=1):
    if not audio_bytes:
        return "", lang
    audio = np.frombuffer(audio_bytes, dtype=np.int16).astype(np.float32) / 32768.0
    loop = asyncio.get_event_loop()
    def run():
        segments, _ = model.transcribe(audio, language=lang, task="transcribe", beam_size=beam_size)
        return " ".join(s.text.strip() for s in segments).strip()
    try:
        text = await loop.run_in_executor(None, run)
        return text, lang
    except Exception as e:
        logger.exception("transcribe error: %s", e)
        return "", lang

@app.get("/health")
async def health():
    return {"ready": app.state.ready}

@app.websocket("/asr")
async def asr_ws(ws: WebSocket):
    await ws.accept()
    buf = bytearray()
    lang = "vi"
    partial_busy = [False]
    logger.info("ASR client connected.")
    try:
        while True:
            msg = await ws.receive()
            if msg.get("type") == "websocket.disconnect":
                break
            if msg.get("bytes") is not None:
                buf.extend(msg["bytes"])
                if len(buf) >= INITIAL_TRANSCRIBE_BYTES and not partial_busy[0]:
                    partial_busy[0] = True
                    audio = bytes(buf)
                    async def _p(audio=audio, lang=lang):
                        try:
                            text, _ = await _transcribe(audio, lang, beam_size=1)
                            if text:
                                await ws.send_text(json.dumps({"type":"partial","text":text}))
                        except Exception as e:
                            logger.exception("partial error: %s", e)
                        finally:
                            partial_busy[0] = False
                    asyncio.create_task(_p())
            elif msg.get("text"):
                try:
                    data = json.loads(msg["text"])
                except json.JSONDecodeError:
                    continue
                action = data.get("action")
                if action == "start":
                    buf.clear()
                    lang = data.get("lang", "vi")
                elif action == "finalize":
                    text, det = await _transcribe(bytes(buf), lang, beam_size=5)
                    await ws.send_text(json.dumps({"type":"final","text":text,"lang":det}))
                    buf.clear()
    except WebSocketDisconnect:
        pass
    except Exception as e:
        logger.exception("asr_ws error: %s", e)
    finally:
        logger.info("ASR client disconnected.")

@app.api_route("/api/{path:path}", methods=["GET", "POST"])
async def proxy_ollama(request: Request, path: str):
    url = f"{OLLAMA_URL}/api/{path}"
    body = await request.body()
    headers = {"Content-Type": request.headers.get("content-type", "application/json")}
    async def stream():
        async with httpx.AsyncClient(timeout=None) as c:
            async with c.stream(request.method, url, content=body, headers=headers) as r:
                async for chunk in r.aiter_raw():
                    yield chunk
    # Cache-Control + X-Accel-Buffering hint các proxy (Cloudflare) đừng buffer
    # streaming NDJSON — nếu không backend chờ dòng đầu mãi -> "đang dịch…" treo.
    return StreamingResponse(stream(), media_type="application/x-ndjson",
        headers={"Cache-Control":"no-cache","X-Accel-Buffering":"no"})
'''
open('/content/asr_server.py','w').write(server_code)
print('Wrote /content/asr_server.py (ASR + Ollama proxy)')

In [ ]:
#@title 4. Khởi động uvicorn (nền)
import subprocess, os, time
env = os.environ.copy()
env['PYTHONPATH'] = '/content'
logf = open('/content/asr_server.log','w')
server_proc = subprocess.Popen(
    ['uvicorn','asr_server:app','--host','0.0.0.0','--port','8000'],
    cwd='/content', env=env, stdout=logf, stderr=subprocess.STDOUT)
print('uvicorn PID:', server_proc.pid)
print('(đang load whisper-medium, mất ~1-2 phút lần đầu do download model)')

In [ ]:
#@title 5. Xem log server (chạy lại đến khi thấy "ASR ready")
import time; time.sleep(2)
print(open('/content/asr_server.log').read()[-2500:])

In [ ]:
#@title 6. Cài + chạy Ollama (GPU), pull model dịch
import subprocess, time, urllib.request, shutil, os

# === Đổi model dịch tại đây ===
MT_MODEL_NAME = 'qwen2.5:7b'  # vd: 'qwen2.5:3b', 'gemma3:4b', 'qwen2.5:14b'

# zstd cần để ollama pull giải nén blob
subprocess.run(['apt-get','update','-qq'], check=False)
subprocess.run(['apt-get','install','-y','zstd'], check=False)
# install.sh: bỏ qua exit code (Colab không có systemd -> script trả non-zero nhưng binary vẫn cài)
print('Running ollama install.sh ...')
subprocess.run('curl -fsSL https://ollama.com/install.sh | sh', shell=True, check=False)
for _ in range(30):
    if shutil.which('ollama') or os.path.exists('/usr/local/bin/ollama'):
        break
    time.sleep(1)
ollama_bin = shutil.which('ollama') or '/usr/local/bin/ollama'
if not os.path.exists(ollama_bin):
    raise SystemExit('Cài ollama thất bại - không thấy binary.')
print('ollama binary:', ollama_bin)
# Chạy server nền (tự detect GPU)
ollama_proc = subprocess.Popen(['ollama','serve'],
    stdout=open('/content/ollama.log','w'), stderr=subprocess.STDOUT)
time.sleep(6)
try:
    r = urllib.request.urlopen('http://127.0.0.1:11434/api/tags', timeout=10)
    print('Ollama server UP:', r.status)
except Exception as e:
    print('Ollama chưa lên. Log:'); print(open('/content/ollama.log').read()[-2000:])
    raise SystemExit(str(e))
print('Pulling', MT_MODEL_NAME, '...')
subprocess.run(['ollama','pull', MT_MODEL_NAME], check=True)
print('Ollama ready at 127.0.0.1:11434')
res = subprocess.run(['ollama','ps'], capture_output=True, text=True)
print(res.stdout)

In [ ]:
#@title 7. Cloudflare Tunnel (cloudflared) — KHÔNG cần token
import subprocess, time, re, os

CF_BIN = '/usr/local/bin/cloudflared'
if not os.path.exists(CF_BIN):
    print('Downloading cloudflared ...')
    for attempt in range(4):
        r = subprocess.run(['curl','-L','--retry','3','--retry-delay','2','-f',
            'https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64',
            '-o', CF_BIN])
        if r.returncode == 0:
            break
        print(f'download attempt {attempt+1} failed (exit {r.returncode}), retry...')
        time.sleep(3)
    if r.returncode != 0:
        print('Fallback to pinned release ...')
        subprocess.run(['curl','-L','-f',
            'https://github.com/cloudflare/cloudflared/releases/download/2025.7.1/cloudflared-linux-amd64',
            '-o', CF_BIN], check=True)
    subprocess.run(['chmod','+x', CF_BIN], check=True)
else:
    print('cloudflared đã có, skip download.')

try:
    cf_proc.send_signal(15)
    time.sleep(2)
except Exception:
    pass
cf_log = open('/content/cloudflared.log','w')
cf_proc = subprocess.Popen([CF_BIN,'tunnel','--url','http://localhost:8000'],
    stdout=cf_log, stderr=subprocess.STDOUT)

base_url = None
for _ in range(60):
    time.sleep(1)
    try:
        txt = open('/content/cloudflared.log').read()
    except Exception:
        txt = ''
    m = re.search(r'https://[a-z0-9-]+\.trycloudflare\.com', txt)
    if m:
        base_url = m.group(0); break
if not base_url:
    print('Không lấy được URL. Log:'); print(open('/content/cloudflared.log').read()[-2000:])
    raise SystemExit('Tunnel chưa sẵn sàng.')

ASR_URL = base_url.replace('https://','wss://') + '/asr'
MT_BASE_URL = base_url
print('HTTP health:', base_url + '/health')
print()
print('>>> ASR_REMOTE_URL cho .env:'); print(ASR_URL)
print()
print('>>> MT_BASE_URL cho .env:'); print(MT_BASE_URL)
print('>>> MT_MODEL cho .env:', MT_MODEL_NAME)

In [ ]:
#@title 8. Test nhanh ASR WS
import numpy as np, asyncio, json, websockets, re
txt = open('/content/cloudflared.log').read()
url = re.search(r'https://[a-z0-9-]+\.trycloudflare\.com', txt).group(0).replace('https://','wss://')+'/asr'
async def test():
    sr=16000; t=np.arange(int(sr*2))/sr
    tone=(0.3*np.sin(2*np.pi*440*t)*32767).astype(np.int16).tobytes()
    async with websockets.connect(url, max_size=None) as ws:
        await ws.send(json.dumps({'action':'start','lang':'vi'}))
        for i in range(0, len(tone), 1600):
            await ws.send(tone[i:i+1600]); await asyncio.sleep(0.05)
        await ws.send(json.dumps({'action':'finalize'}))
        while True:
            raw=await asyncio.wait_for(ws.recv(), timeout=30)
            print('<=', raw[:200])
            if json.loads(raw).get('type')=='final': break
await test()
print('OK — WS ASR hoạt động qua Cloudflare.')

## Dùng với backend local

Sau khi cell 7 in ra 3 giá trị, điền vào `D:/VNAI/.env`:

```env
ASR_REMOTE_URL=wss://xxx-yyy.trycloudflare.com/asr
MT_BASE_URL=https://xxx-yyy.trycloudflare.com
MT_MODEL=qwen2.5:7b   # đúng với MT_MODEL_NAME ở cell 6
```

`MT_BASE_URL` = cùng host Cloudflare (proxy `/api/*` forward sang Ollama local). `MT_MODEL` = đúng tên model đã `ollama pull` ở cell 6.

Chạy backend local:
```powershell
python -m uvicorn backend.main:app --host 0.0.0.0 --port 8000
```

- `ASR_REMOTE_URL` set → ASR gọi Colab GPU. Để trống → whisper-medium local CPU.
- `MT_BASE_URL` set → dịch gọi Ollama trên Colab GPU. Để trống → Ollama local (qwen2.5:1.5b).
- VAD + TTS vẫn chạy local.

> URL Cloudflare đổi mỗi lần chạy cell 7 → cập nhật `.env` + restart backend mỗi lần Colab reconnect.

In [ ]:
#@title Dừng tất cả (uvicorn + ollama + cloudflared)
import signal
for name, var in [('uvicorn','server_proc'),('ollama','ollama_proc'),('cloudflared','cf_proc')]:
    try:
        var.send_signal(signal.SIGTERM); print(name, 'stopped')
    except Exception as e:
        print(name, '?', e)